# Phase 3 — Predictive Modeling

Two models are compared for predicting sale price:
- **Baseline:** Ridge regression (linear, with scaling)
- **Main:** Gradient Boosting Regressor

Evaluation uses an 80/20 train-test split plus 5-fold cross-validation on the training set.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import features as feat_module
import model as model_module

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

## 1. Prepare features

In [ ]:
X, y, encoders = feat_module.run()
print(f'Feature matrix: {X.shape}')
print(f'Target — median: ${y.median():,.0f}, mean: ${y.mean():,.0f}')
X.head(3)

## 2. Train and evaluate both models

In [ ]:
results = model_module.run(X, y)

summary = pd.DataFrame(results['results'])
summary[['model', 'test_rmse', 'test_r2', 'cv_rmse_mean', 'cv_rmse_std']]

The Gradient Boosting model substantially outperforms the linear baseline on both RMSE and R². Cross-validation confirms the improvement is not due to overfitting: the CV and test RMSEs are consistent.

## 3. Predicted vs. actual — both models

In [ ]:
y_test = results['y_test']
y_pred_gb = results['y_pred_gb']
y_pred_ridge = results['y_pred_ridge']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, name in [
    (axes[0], y_pred_ridge, 'Ridge (baseline)'),
    (axes[1], y_pred_gb,    'Gradient Boosting')
]:
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.scatter(y_test, preds, alpha=0.3, s=12, color='steelblue')
    ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Actual price ($)')
    ax.set_ylabel('Predicted price ($)')
    ax.set_title(name)
    ax.legend(fontsize=9)

plt.suptitle('Predicted vs. actual sale price', y=1.01)
plt.tight_layout()
plt.show()

Ridge regression shows a noticeable fan shape: it overestimates cheap homes and underestimates expensive ones, a sign of regression-to-the-mean in linear models. Gradient Boosting predictions track the diagonal much more closely, though both models underestimate the highest-priced properties (above $400k), where training examples are sparse.

## 4. Residual analysis

In [ ]:
residuals_gb = y_test - y_pred_gb

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred_gb, residuals_gb, alpha=0.3, s=12, color='steelblue')
axes[0].axhline(0, color='firebrick', linewidth=1, linestyle='--')
axes[0].set_xlabel('Predicted price ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Residuals vs. predicted (Gradient Boosting)')

axes[1].hist(residuals_gb, bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='firebrick', linewidth=1, linestyle='--')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual distribution')

plt.tight_layout()
plt.show()

Residuals are approximately symmetric around zero with no strong systematic pattern, which is consistent with a well-fit model. The right tail of the residual histogram reflects the high-price homes where the model systematically under-predicts.

## 5. Feature importance

In [ ]:
importance_df = results['feature_importance']
top20 = importance_df.head(20)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top20['feature'][::-1], top20['importance'][::-1], color='steelblue', alpha=0.85)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('Top 20 features — Gradient Boosting')
plt.tight_layout()
plt.show()

`overall_qual` dominates feature importance, followed by `gr_liv_area` and `age_at_sale`. This aligns with the correlations found in the EDA. Neighborhood, while important conceptually, ranks lower here because it is label-encoded — a tree-based model cannot fully exploit ordinal relationships in label-encoded categoricals. One-hot encoding would likely push neighborhood importance higher.

## 6. Model limitations

**What the model does well:** For the middle 80% of the price distribution ($100k–$350k), predictions are generally within ±$25k. The model generalizes well given the train/test consistency.

**Known limitations:**
- The model systematically under-predicts luxury homes (>$400k) because fewer than 50 training examples exist in that range. More data or a separate high-end model would be needed.
- Categorical features are label-encoded, which imposes an arbitrary ordinal relationship on neighborhoods. Switching to one-hot or target encoding would improve the neighborhood signal.
- The dataset covers 2006–2010 in a single mid-size Midwestern city. The model should not be applied to other geographies or time periods without retraining.
- External factors (interest rates, local employment, school quality) are absent from the feature set and cannot be predicted from property characteristics alone.

In [ ]:
# Summary table
summary_fmt = summary.copy()
for col in ['test_rmse', 'cv_rmse_mean', 'cv_rmse_std']:
    summary_fmt[col] = summary_fmt[col].apply(lambda v: f'${v:,.0f}')
summary_fmt['test_r2'] = summary_fmt['test_r2'].apply(lambda v: f'{v:.3f}')
summary_fmt.columns = ['Model', 'Test RMSE', 'Test R²', 'CV RMSE (mean)', 'CV RMSE (std)']
summary_fmt